# Kaggle video database annotation notebook

This notebook is modeled after `handwash_inference_timeline_local.ipynb`, but it is focused on **batch annotation of the full Kaggle video database**.

It does the following:
- downloads / scans the Kaggle WHO6-style database
- loads a trained Keras model
- runs inference on **every frame** (or every `FRAME_STRIDE` frames) of each video
- writes a **per-frame CSV** with:
  - video path
  - frame index
  - timestamp
  - ground-truth label
  - predicted class
  - predicted confidence
  - probability for every class
- writes an **annotated output video** for each input video
- writes a **dataset-level manifest CSV** summarizing all processed videos

> Assumption: the ground-truth label for a video is the class implied by the Kaggle folder name. If you have frame-level ground-truth annotations, you can replace the `gt_ids` construction in `annotate_single_video`.


In [1]:
%pip install -q numpy pandas opencv-python tensorflow matplotlib tqdm requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import math
import json
import random
import tarfile
import importlib.util
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import Video, display


/Users/alinikkhah/Documents/Work/Handwash/env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration
Adjust these values before running the notebook.

In [ ]:
# -----------------------------
# Dataset
# -----------------------------
KAGGLE_URL = "https://github.com/atiselsts/data/raw/master/kaggle-dataset-6classes.tar"
KAGGLE_ARCHIVE = Path("datasets/raw/kaggle/kaggle-dataset-6classes.tar")
KAGGLE_ROOT = Path("datasets/raw/kaggle/kaggle-dataset-6classes")

# -----------------------------
# Model
# -----------------------------
MODEL_PATH = Path("/Users/alinikkhah/Documents/Work/Handwash/inference/mobilenetv2_final.keras")

# -----------------------------
# Output
# -----------------------------
OUTPUT_ROOT = Path("inference/outputs/kaggle_full_annotation")
ANNOTATED_VIDEO_DIR = OUTPUT_ROOT / "annotated_videos"
TIMELINE_DIR = OUTPUT_ROOT / "timelines"
MANIFEST_PATH = OUTPUT_ROOT / "dataset_manifest.csv"

# -----------------------------
# Inference behavior
# -----------------------------
FRAME_STRIDE = 1            # use 1 for every frame, 2/3/... to speed up processing
BATCH_SIZE = 32
MAX_VIDEOS = None           # set to int for debugging, e.g. 10
MAX_FRAMES_PER_VIDEO = None # set to int for debugging
SMOOTHING_WINDOW = 7        # moving-average smoothing over probabilities
WRITE_ANNOTATED_VIDEOS = True
SHOW_EXAMPLE_OUTPUT = True

# -----------------------------
# Fallback class names / mapping
# Update these if your training config uses different names.
# -----------------------------
DEFAULT_CLASS_NAMES = [
    "Step_1",
    "Step_2",
    "Step_3",
    "Step_4",
    "Step_5",
    "Step_6",
]

DEFAULT_KAGGLE_CLASS_MAPPING = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "class0": 0,
    "class1": 1,
    "class2": 2,
    "class3": 3,
    "class4": 4,
    "class5": 5,
    "step1": 0,
    "step2": 1,
    "step3": 2,
    "step4": 3,
    "step5": 4,
    "step6": 5,
}

np.random.seed(42)
random.seed(42)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ANNOTATED_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
TIMELINE_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_PATH:", MODEL_PATH)
print("KAGGLE_ROOT:", KAGGLE_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


MODEL_PATH: inference/mobilenetv2_final.keras
KAGGLE_ROOT: datasets/raw/kaggle/kaggle-dataset-6classes
OUTPUT_ROOT: inference/outputs/kaggle_full_annotation


## Optional: load training/inference config if it exists

In [4]:
def _load_module(path: Path, name: str):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load module from {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for parent in [start] + list(start.parents):
        if (parent / "inference" / "config.py").exists() or (parent / "training" / "config.py").exists():
            return parent
    return start


REPO_ROOT = find_repo_root()
cfg = None
for cfg_path in [REPO_ROOT / "inference" / "config.py", REPO_ROOT / "training" / "config.py"]:
    if cfg_path.exists():
        cfg = _load_module(cfg_path, "cfg")
        print("Loaded config from", cfg_path)
        break

if cfg is not None:
    CLASS_NAMES = list(getattr(cfg, "CLASS_NAMES", DEFAULT_CLASS_NAMES))
    KAGGLE_CLASS_MAPPING = dict(getattr(cfg, "KAGGLE_CLASS_MAPPING", DEFAULT_KAGGLE_CLASS_MAPPING))
    IMG_SIZE = tuple(getattr(cfg, "IMG_SIZE", (224, 224)))
    SEQUENCE_LENGTH = int(getattr(cfg, "SEQUENCE_LENGTH", 16))
else:
    CLASS_NAMES = list(DEFAULT_CLASS_NAMES)
    KAGGLE_CLASS_MAPPING = dict(DEFAULT_KAGGLE_CLASS_MAPPING)
    IMG_SIZE = (224, 224)
    SEQUENCE_LENGTH = 16

print("CLASS_NAMES:", CLASS_NAMES)
print("IMG_SIZE:", IMG_SIZE)


Loaded config from /Users/alinikkhah/Documents/Work/Handwash/inference/config.py
CLASS_NAMES: ['Other', 'Step1_PalmToPalm', 'Step2_PalmOverDorsum', 'Step3_InterlacedFingers', 'Step4_BackOfFingers', 'Step5_ThumbRub', 'Step6_Fingertips']
IMG_SIZE: (224, 224)


## Dataset download / extraction

In [5]:
import requests


def download_with_progress(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print("Archive already exists:", dest)
        return dest

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                pbar.update(len(chunk))
    return dest


def extract_tar(archive_path: Path, target_dir: Path):
    if target_dir.exists() and any(target_dir.iterdir()):
        print("Dataset already extracted:", target_dir)
        return target_dir

    target_dir.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r") as tar:
        tar.extractall(target_dir.parent)
    return target_dir


download_with_progress(KAGGLE_URL, KAGGLE_ARCHIVE)
extract_tar(KAGGLE_ARCHIVE, KAGGLE_ROOT)
print("Ready:", KAGGLE_ROOT)


kaggle-dataset-6classes.tar: 100%|██████████| 1.30G/1.30G [06:02<00:00, 3.58MB/s]
/var/folders/y5/4ntn99k92d70gksjt9w9jgzr0000gn/T/ipykernel_4254/2462620678.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(target_dir.parent)


Ready: datasets/raw/kaggle/kaggle-dataset-6classes


## Scan videos in the Kaggle database

In [6]:
VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")


def kaggle_class_id_from_folder(name: str):
    name_lower = name.lower().strip()
    if name_lower in KAGGLE_CLASS_MAPPING:
        return int(KAGGLE_CLASS_MAPPING[name_lower])

    digits = "".join(ch for ch in name_lower if ch.isdigit())
    if digits:
        value = int(digits)
        if value in range(len(CLASS_NAMES)):
            return value
        if (value - 1) in range(len(CLASS_NAMES)):
            return value - 1
    return None


def scan_kaggle_videos(root: Path):
    records = []
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        class_id = kaggle_class_id_from_folder(class_dir.name)
        if class_id is None:
            print(f"Skipping unmapped folder: {class_dir.name}")
            continue
        for video_path in sorted(class_dir.rglob("*")):
            if video_path.suffix.lower() not in VIDEO_EXTS:
                continue
            records.append({
                "video_path": str(video_path),
                "class_id": int(class_id),
                "class_name": CLASS_NAMES[int(class_id)],
                "folder_name": class_dir.name,
            })
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(["class_id", "video_path"]).reset_index(drop=True)
    return df


videos_df = scan_kaggle_videos(KAGGLE_ROOT)
if MAX_VIDEOS is not None:
    videos_df = videos_df.head(MAX_VIDEOS).copy()

print("Total videos:", len(videos_df))
display(videos_df.head())
print(videos_df["class_name"].value_counts(dropna=False).sort_index())


Total videos: 300


,video_path,class_id,class_name,folder_name
0,datasets/raw/kaggle/kaggle-dataset-6classes/0/...,0,Other,0
1,datasets/raw/kaggle/kaggle-dataset-6classes/0/...,0,Other,0
2,datasets/raw/kaggle/kaggle-dataset-6classes/0/...,0,Other,0
3,datasets/raw/kaggle/kaggle-dataset-6classes/0/...,0,Other,0
4,datasets/raw/kaggle/kaggle-dataset-6classes/0/...,0,Other,0


class_name
Other                      50
Step1_PalmToPalm           25
Step2_PalmOverDorsum       50
Step3_InterlacedFingers    25
Step4_BackOfFingers        50
Step5_ThumbRub             50
Step6_Fingertips           50
Name: count, dtype: int64


## Show a sample input video

In [7]:
if not videos_df.empty:
    row = videos_df.sample(1, random_state=42).iloc[0]
    print(row["video_path"], "| GT:", row["class_name"])
    display(Video(row["video_path"], embed=True, width=360))


datasets/raw/kaggle/kaggle-dataset-6classes/5/HandWash_002_A_08_G_01.mp4 | GT: Step5_ThumbRub


## Load model

In [8]:
custom_objects = {}

try:
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_v2_preprocess
    custom_objects["preprocess_input"] = mobilenet_v2_preprocess
except Exception:
    pass

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects=custom_objects,
    compile=False,
    safe_mode=False,
)
model.summary()


ValueError: File not found: filepath=inference/mobilenetv2_final.keras. Please ensure the file is an accessible `.keras` zip file.

## Inference helpers

In [ ]:
def preprocess_frame(frame_rgb, img_size=IMG_SIZE):
    resized = cv2.resize(frame_rgb, img_size)
    x = resized.astype(np.float32) / 255.0
    return x


def collect_video_frames(video_path, frame_stride=1, max_frames=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 30.0

    inputs = []
    frames_bgr = []
    frame_indices = []
    timestamps = []

    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % frame_stride == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            inputs.append(preprocess_frame(frame_rgb))
            frames_bgr.append(frame.copy())
            frame_indices.append(idx)
            timestamps.append(idx / fps)
            if max_frames is not None and len(inputs) >= max_frames:
                break
        idx += 1

    cap.release()
    inputs = np.stack(inputs, axis=0) if inputs else np.zeros((0, *IMG_SIZE, 3), dtype=np.float32)
    return float(fps), np.array(frame_indices), np.array(timestamps, dtype=np.float32), inputs, frames_bgr


def _get_model_input_shape(model):
    shape = model.input_shape
    if isinstance(shape, (list, tuple)) and shape and isinstance(shape[0], (list, tuple)):
        shape = shape[0]
    return shape


def is_sequence_model(model):
    shape = _get_model_input_shape(model)
    return shape is not None and len(shape) == 5


def get_sequence_length(model, fallback=SEQUENCE_LENGTH):
    shape = _get_model_input_shape(model)
    if shape is not None and len(shape) >= 2 and shape[1]:
        return int(shape[1])
    return int(fallback)


def predict_batches(model, inputs, batch_size=32):
    if len(inputs) == 0:
        return np.zeros((0, len(CLASS_NAMES)), dtype=np.float32)
    preds = []
    for i in range(0, len(inputs), batch_size):
        preds.append(model.predict(inputs[i:i+batch_size], verbose=0))
    return np.concatenate(preds, axis=0)


def predict_sequence_probs(model, inputs, seq_len=None, stride=1, batch_size=8):
    if len(inputs) == 0:
        return np.zeros((0, len(CLASS_NAMES)), dtype=np.float32)

    seq_len = int(seq_len or get_sequence_length(model))
    stride = max(1, int(stride))

    if len(inputs) < seq_len:
        pad_len = seq_len - len(inputs)
        pad = np.repeat(inputs[-1][None, ...], pad_len, axis=0)
        sequences = np.expand_dims(np.concatenate([inputs, pad], axis=0), axis=0)
        starts = [0]
    else:
        starts = list(range(0, len(inputs) - seq_len + 1, stride))
        sequences = np.stack([inputs[s:s+seq_len] for s in starts], axis=0)

    preds = []
    for i in range(0, len(sequences), batch_size):
        preds.append(model.predict(sequences[i:i+batch_size], verbose=0))
    preds = np.concatenate(preds, axis=0) if preds else np.zeros((0, len(CLASS_NAMES)), dtype=np.float32)

    probs = np.zeros((len(inputs), preds.shape[1] if len(preds) else len(CLASS_NAMES)), dtype=np.float32)
    counts = np.zeros((len(inputs), 1), dtype=np.float32)
    for pred, start in zip(preds, starts):
        end = min(start + seq_len, len(inputs))
        probs[start:end] += pred
        counts[start:end] += 1.0
    probs = probs / np.clip(counts, 1e-6, None)
    return probs


def predict_model_probs(model, inputs, batch_size=32):
    if is_sequence_model(model):
        seq_len = get_sequence_length(model)
        return predict_sequence_probs(model, inputs, seq_len=seq_len, stride=1, batch_size=max(1, batch_size // 4))
    return predict_batches(model, inputs, batch_size=batch_size)


def smooth_probs_moving_avg(probs, window=7):
    if len(probs) == 0 or window <= 1:
        return probs
    kernel = np.ones(window, dtype=np.float32) / float(window)
    out = np.zeros_like(probs)
    for c in range(probs.shape[1]):
        out[:, c] = np.convolve(probs[:, c], kernel, mode="same")
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-6, None)
    return out


## Output helpers: timeline CSV + annotated video

In [ ]:
def draw_text_block(
    frame,
    lines,
    origin=(10, 30),
    line_height=22,
    font_scale=0.6,
    thickness=1,
    text_color=(255, 255, 255),
    bg_color=(0, 0, 0),
):
    x, y = origin
    font = cv2.FONT_HERSHEY_SIMPLEX
    for line in lines:
        (text_w, text_h), baseline = cv2.getTextSize(line, font, font_scale, thickness)
        cv2.rectangle(frame, (x - 4, y - text_h - 4), (x + text_w + 4, y + baseline + 4), bg_color, -1)
        cv2.putText(frame, line, (x, y), font, font_scale, text_color, thickness, cv2.LINE_AA)
        y += line_height


def make_probability_columns(probs, prefix="prob"):
    cols = {}
    for i, class_name in enumerate(CLASS_NAMES):
        safe_name = str(class_name).replace(" ", "_")
        cols[f"{prefix}_{safe_name}"] = probs[:, i] if len(probs) else []
    return cols


def build_timeline_df(video_path, frame_indices, timestamps, gt_ids, pred_ids, probs):
    pred_conf = probs.max(axis=1) if len(probs) else np.array([], dtype=np.float32)
    df = pd.DataFrame({
        "video_path": str(video_path),
        "frame_index": frame_indices.astype(int),
        "timestamp_s": timestamps.astype(float),
        "gt_class_id": gt_ids.astype(int),
        "gt_class_name": [CLASS_NAMES[int(x)] for x in gt_ids],
        "pred_class_id": pred_ids.astype(int),
        "pred_class_name": [CLASS_NAMES[int(x)] for x in pred_ids],
        "pred_confidence": pred_conf.astype(float),
    })
    for col, values in make_probability_columns(probs, prefix="prob").items():
        df[col] = values
    return df


def write_annotated_video(frames_bgr, timestamps, gt_ids, pred_ids, probs, out_path, fps_out):
    if not frames_bgr:
        return None

    h, w = frames_bgr[0].shape[:2]
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps_out, (w, h))
    for i, frame in enumerate(frames_bgr):
        pred_id = int(pred_ids[i])
        gt_id = int(gt_ids[i])
        conf = float(probs[i, pred_id]) if len(probs) else float("nan")

        left_lines = [
            f"t = {timestamps[i]:.2f}s",
            f"GT   : {CLASS_NAMES[gt_id]}",
            f"Pred : {CLASS_NAMES[pred_id]}",
            f"Conf : {conf:.3f}",
        ]
        draw_text_block(frame, left_lines, origin=(10, 30))

        right_lines = []
        for j, class_name in enumerate(CLASS_NAMES):
            marker = ">" if j == pred_id else " "
            right_lines.append(f"{marker} {class_name}: {probs[i, j]:.3f}")
        draw_text_block(frame, right_lines, origin=(max(10, w - 300), 30))
        writer.write(frame)

    writer.release()
    return out_path


## Main pipeline for one video

In [ ]:
def annotate_single_video(row, output_root=OUTPUT_ROOT):
    video_path = Path(row["video_path"])
    gt_class_id = int(row["class_id"])

    fps, frame_indices, timestamps, inputs, frames_bgr = collect_video_frames(
        video_path,
        frame_stride=FRAME_STRIDE,
        max_frames=MAX_FRAMES_PER_VIDEO,
    )
    if len(inputs) == 0:
        return None

    probs_raw = predict_model_probs(model, inputs, batch_size=BATCH_SIZE)
    probs = smooth_probs_moving_avg(probs_raw, window=SMOOTHING_WINDOW)
    pred_ids = np.argmax(probs, axis=1).astype(np.int32)
    gt_ids = np.full(len(pred_ids), gt_class_id, dtype=np.int32)

    video_stem = video_path.stem
    timeline_path = TIMELINE_DIR / f"{video_stem}_timeline.csv"
    annotated_video_path = ANNOTATED_VIDEO_DIR / f"{video_stem}_annotated.mp4"

    timeline_df = build_timeline_df(
        video_path=video_path,
        frame_indices=frame_indices,
        timestamps=timestamps,
        gt_ids=gt_ids,
        pred_ids=pred_ids,
        probs=probs,
    )
    timeline_df.to_csv(timeline_path, index=False)

    correct = (gt_ids == pred_ids)
    frame_accuracy = float(correct.mean()) if len(correct) else float("nan")

    if WRITE_ANNOTATED_VIDEOS:
        fps_out = fps / FRAME_STRIDE if FRAME_STRIDE else fps
        write_annotated_video(
            frames_bgr=frames_bgr,
            timestamps=timestamps,
            gt_ids=gt_ids,
            pred_ids=pred_ids,
            probs=probs,
            out_path=annotated_video_path,
            fps_out=fps_out,
        )
    else:
        annotated_video_path = None

    return {
        "video_path": str(video_path),
        "class_id": gt_class_id,
        "class_name": CLASS_NAMES[gt_class_id],
        "num_frames_written": int(len(timeline_df)),
        "frame_accuracy": frame_accuracy,
        "timeline_csv": str(timeline_path),
        "annotated_video": str(annotated_video_path) if annotated_video_path else "",
        "pred_majority_class": CLASS_NAMES[int(timeline_df["pred_class_id"].mode().iloc[0])],
        "mean_pred_confidence": float(timeline_df["pred_confidence"].mean()),
    }


## Run on the whole Kaggle video database

In [ ]:
manifest_rows = []

for row in tqdm(list(videos_df.to_dict(orient="records")), desc="Annotating videos"):
    result = annotate_single_video(row)
    if result is not None:
        manifest_rows.append(result)

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print("Finished.")
print("Manifest saved to:", MANIFEST_PATH)
display(manifest_df.head())
print("Processed videos:", len(manifest_df))


## Inspect results

In [ ]:
if not manifest_df.empty:
    display(manifest_df.groupby("class_name")[["num_frames_written", "frame_accuracy", "mean_pred_confidence"]].mean())

    sample_out = manifest_df.iloc[0]
    print("Timeline CSV:", sample_out["timeline_csv"])
    display(pd.read_csv(sample_out["timeline_csv"]).head())

    if SHOW_EXAMPLE_OUTPUT and WRITE_ANNOTATED_VIDEOS and sample_out["annotated_video"]:
        print("Annotated video:", sample_out["annotated_video"])
        display(Video(sample_out["annotated_video"], embed=True, width=360))


## Notes

- This notebook assumes **video-level labels** from the Kaggle folder names, so each frame in a video receives the same GT label.
- If your model expects a specific preprocessing function beyond `/255.0`, replace `preprocess_frame`.
- If your model is sequence-based (LSTM/GRU/TCN with shape `[batch, time, h, w, c]`), the notebook already supports that automatically.
- To reduce runtime, increase `FRAME_STRIDE` or set `MAX_VIDEOS` while debugging.
- Output structure:
  - `inference/outputs/kaggle_full_annotation/timelines/*.csv`
  - `inference/outputs/kaggle_full_annotation/annotated_videos/*.mp4`
  - `inference/outputs/kaggle_full_annotation/dataset_manifest.csv`
